# H&M 2년 M1 후보상품 관계 진단
기존 seed-42 M1 체크포인트에서 Dunnhumby와 동일한 후보상품 관계를 확인합니다. 재학습·최종 test·holdout은 수행하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '909a8b2'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip().startswith(REVIEWED_SHA)

In [ ]:
import importlib
import inspect
import json
import torch
import clv_m3_clv_conditioned_category_transition_graph as transition_graph
transition_graph = importlib.reload(transition_graph)
import lightgcn_clv_candidate_relation_diagnostic as candidate_diagnostic
candidate_diagnostic = importlib.reload(candidate_diagnostic)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
basket_source = inspect.getsource(transition_graph._basket_order)
assert 'if "b_raw" in train' in basket_source and '"_basket_id"' in basket_source, (
    '수정 전 전이 모듈이 메모리에 남아 있습니다. 런타임을 다시 시작하고 모두 실행하세요.'
)
assert candidate_diagnostic.CODE_VERSION == 'm1-clv-candidate-relation-diagnostic-v1'
cfg = candidate_diagnostic.configure_candidate_relation_diagnostic('hm')
print(json.dumps(candidate_diagnostic.preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
transition_graph = importlib.reload(transition_graph)
candidate_diagnostic = importlib.reload(candidate_diagnostic)
cfg = candidate_diagnostic.configure_candidate_relation_diagnostic('hm')
paths = candidate_diagnostic.run_candidate_relation_diagnostic(cfg)

In [ ]:
import pandas as pd
from IPython.display import display

summary = pd.read_csv(paths['relation_summary_csv'])
focus = summary.group_type.isin(['overall', 'fixed_clv_segment', 'high_clv_composition'])
columns = [
    'signal', 'group_type', 'group', 'n_users', 'candidate_pair_count',
    'pair_balanced_win_rate', 'pair_strict_win_rate', 'pair_tie_rate',
    'mean_pair_score_difference',
]
print('1) 누락 정답이 M1 Top-10 오추천을 이기는 비율')
display(summary.loc[focus, columns])
print('판독 기준: 같은 관계가 Dunnhumby와 H&M 모두에서 전체·고CLV 승률 0.5 초과')
print('결과 파일:', paths)